# Skip Prediction

Predict whether a Spotify listening event will be skipped using supervised machine learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

processed = BASE_DIR / 'data' / 'processed'
figures = BASE_DIR / 'reports' / 'figures'
models_dir = BASE_DIR / 'models'

figures.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(processed / 'spotify_clean_history.csv')

# Target must be True/False
df['skipped'] = df['skipped'].fillna(False).astype(bool)

# Make sure these columns have the expected types
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')
df['is_weekend'] = df['is_weekend'].fillna(False).astype(bool)
df['shuffle'] = df['shuffle'].fillna(False).astype(bool)

# Remove rows where the target is unavailable
df = df.dropna(subset=['skipped']).reset_index(drop=True)

print('Rows:', len(df))
print('Skip rate:', round(df['skipped'].mean() * 100, 2), '%')
print(df['skipped'].value_counts())


## 1. Choose input features and target

The model uses listening context available for the event. We do not use `minutes_played` because that is known after the listening event and could leak information about the target.

In [ ]:
features = [
    'hour',
    'day_of_week',
    'is_weekend',
    'shuffle',
    'reason_start',
    'master_metadata_album_artist_name'
]

target = 'skipped'

X = df[features].copy()
y = df[target].copy()

print(X.head())
print('\nData types:')
print(X.dtypes)


## 2. Split into training and test data

The model learns from the training set and is evaluated on the unseen test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))


## 3. Prepare the features

Day of week and Spotify reason fields are categorical. One-hot encoding converts them into numeric columns.

In [ ]:
categorical_features = [
    'day_of_week',
    'reason_start',
    'master_metadata_album_artist_name'
]

numeric_features = [
    'hour',
    'is_weekend',
    'shuffle'
]

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=5))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_features),
    ('numeric', numeric_pipeline, numeric_features)
])


## 4. Train Logistic Regression

Logistic Regression is used as a simple classification baseline.

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

model.fit(X_train, y_train)
print('Model training completed.')


## 5. Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print('Accuracy:', round(accuracy, 3))
print('\nClassification report:')
print(classification_report(
    y_test,
    y_pred,
    target_names=['Not skipped', 'Skipped'],
    zero_division=0
))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not skipped', 'Skipped']
)
disp.plot()
plt.title('Skip Prediction Confusion Matrix')
plt.tight_layout()
plt.savefig(figures / 'skip_prediction_confusion_matrix.png')
plt.show()


## 6. Save the trained model

In [ ]:
model_path = models_dir / 'skip_prediction_model.joblib'
joblib.dump(model, model_path)
print('Saved:', model_path)


## Final observations

- Target: skipped
- Model: Logistic Regression
- Accuracy: 0.642
- Precision for skipped: 0.39 
- Recall for skipped: 0.64 
- F1-score for skipped: 0.48
- Main limitation: this is a baseline model using listening-context features.